# BAF EDA Gap Matrix
In designing agentic systems for identifying fraud risk among bank account applications, choosing the right model for calculating risk scores can be overwhelming. XGBoost or Neural Nets? To preprocess or not? What if the data is represented as graphs? This notebook will walk you through comparative setups that hope to answer these questions.

Let us first set up the dependencies and configurations, which shall be explained in time. 

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eda.baf_variants import (
    attach_lift,
    compute_variant_base_rates,
    official_eval_paths,
    suite_mapping_table,
)
from src.eda.feature_plots import draw_features_baf, list_baf_feature_columns
from src.graph.property_graph import (
    build_property_graph,
    property_graph_embeddings,
    property_graph_stats,
)
from src.graph.temporal_knn import (
    TemporalKNNConfig,
    build_temporal_knn_graph,
    mean_neighbor_features,
    temporal_knn_stats,
)
from src.modeling.cost_metrics import (
    COST_DISCLAIMER,
    CostConfig,
    assumptions_dict,
    build_cutoff_metrics_table,
    build_false_alarm_summary,
    compute_exposure_tp_per_account,
    cost_check_fp_per_account,
    cost_unlock_fp_per_account,
    maximize_profit,
    plot_cutoff_economics,
    plot_false_alarm_rate,
)
from src.modeling.metrics import best_f1_threshold, calibrate_platt, evaluate_binary_classifier
from src.modeling.train_vanilla import train_vanilla
from src.modeling.xgb_runtime import resolve_xgb_compute
from src.preprocessing.baf_preprocessor import BAFPreprocessor, TimeSplit
from xgboost import XGBClassifier

COST_CONFIG = CostConfig(
    clerk_salary_annual=400_000,
    hours_per_month=160,
    hours_to_unlock_fp=32,
    hours_to_check_fp=8,
    fraud_volume_proxy_col="intended_balcon_amount",
    exposure_unit="dataset_amount_units",
    ops_currency="HKD",
    ops_currency_is_hypothetical=True,
    ops_narrative="HK_demo",
)

CONFIG = {
    "seed": 42,
    "data_dir": REPO_ROOT / "data",
    "data_path": REPO_ROOT / "data" / "base.csv",
    "stage1_variant": REPO_ROOT / "data" / "base.csv",
    "stage2_ablation_variant": REPO_ROOT / "data" / "variant_1.csv",  # Variant I
    "results_dir": REPO_ROOT / "results" / "eda_matrix",
    "knn_k": 20,
    "knn_metric": "cosine",
    "knn_mutual": False,
    "use_smote_stage1": False,
    "use_yeo_johnson": True,
    "recall_at_precision": 0.8,
    "eda_plot_all_features": True,
    "eda_max_plots": None,
    "shap_sample_size": 500,
}

CONFIG["eval_suite"] = official_eval_paths(CONFIG["data_dir"])  # Base + I–V
CONFIG["results_dir"].mkdir(parents=True, exist_ok=True)
np.random.seed(CONFIG["seed"])

print("Repo root:", REPO_ROOT)
print("Results:", CONFIG["results_dir"])
print("COST disclaimer:", COST_DISCLAIMER)
print("\nOfficial BAF mapping:")
print(suite_mapping_table().to_string(index=False))


## Chapter 1: Exploratory Data Analysis

The tabular data used is synthetically generated, in which each instance (row) has properties (column values) that are not influenced by any other instance. You might think, "actual instances of fraudulent applications are likely related to each other," and that would be correct. However, regulations restrict public financial datasets from representing real data. Hence, unless your team has access to raw data through partnerships with financial institutions, you would need to work around this constraint.

One might then ask: what is the point of having graph representations when the datapoints are unrelated by design? In our case, we shall construct a property graph and a temporal kNN graph to see if either (or both) of the two approaches can derive meaningful connections from the BAF dataset.

**Data source.** This notebook uses Feedzai’s NeurIPS 2022 Bank Account Fraud (BAF) suite. Local CSVs under `data/` are the official **Base + Variants I–V** files.

- Dataset: [Bank Account Fraud Dataset Suite (NeurIPS 2022) on Kaggle](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022)
- Official GitHub (paper, datasheet, code): [feedzai/bank-account-fraud](https://github.com/feedzai/bank-account-fraud)

Now, let us inspect all BAF features on the **base variant** (*e.g.* target stats, factor plots, missingness, correlations).

In [ ]:
DATA_PATH = Path(CONFIG["stage1_variant"])
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"BAF CSV not found at {DATA_PATH}. Place base.csv under data/ or update CONFIG."
    )

df = pd.read_csv(DATA_PATH)
numeric_cols, categorical_cols = list_baf_feature_columns(df)
all_feature_cols = categorical_cols + numeric_cols
print("Shape:", df.shape)
print("Features:", len(all_feature_cols), "categorical:", len(categorical_cols), "numeric:", len(numeric_cols))
print("\nFraud rate overall (Base):", df["fraud_bool"].mean())
print("\nFraud rate by month:")
print(df.groupby("month")["fraud_bool"].agg(["count", "mean"]).rename(columns={"mean": "fraud_rate"}))

fig, ax = plt.subplots(figsize=(8, 3))
df.groupby("month")["fraud_bool"].mean().plot(kind="bar", ax=ax, title="Fraud rate by month (Base)")
ax.set_ylabel("fraud_rate")
plt.tight_layout()
plt.show()


In [ ]:
plot_cols = all_feature_cols
if CONFIG.get("eda_max_plots"):
    plot_cols = plot_cols[: CONFIG["eda_max_plots"]]
elif not CONFIG.get("eda_plot_all_features", True):
    plot_cols = plot_cols[:10]
draw_features_baf(df, feature_cols=plot_cols)


### Notes for reading the charts

#### 1. Feature glossary ([Feedzai datasheet](https://github.com/feedzai/bank-account-fraud/blob/main/documents/datasheet.pdf)) 

**Target / time**

| Feature | Type | Definition |
|---------|------|------------|
| `fraud_bool` | binary | Fraud label (1 = fraud, 0 = genuine) |
| `month` | numeric | Application month index in `[0, 7]` |

**Categorical features**

| Feature | Definition | Values |
|---------|------------|--------|
| `payment_type` | Credit payment plan type | 5 anonymized codes: AA–AE |
| `employment_status` | Employment status of the applicant | 7 anonymized codes: CA–CG |
| `housing_status` | Current residential status | 7 anonymized codes: BA–BG |
| `email_is_free` | Email domain is free vs paid | `{0, 1}` |
| `phone_home_valid` | Validity of provided home phone | `{0, 1}` |
| `phone_mobile_valid` | Validity of provided mobile phone | `{0, 1}` |
| `has_other_cards` | Applicant has other cards from the same bank | `{0, 1}` |
| `foreign_request` | Request origin country differs from bank country | `{0, 1}` |
| `source` | Online application channel | `INTERNET` (browser) or `APP` |
| `device_os` | OS of the requesting device | Windows, Macintosh, Linux, X11, other |
| `keep_alive_session` | User option on session logout | `{0, 1}` |

**Numerical features**

| Feature | Definition | Notes / range (datasheet) |
|---------|------------|---------------------------|
| `income` | Annual income in quantiles | `[0, 1]` |
| `name_email_similarity` | Similarity between email and applicant name | `[0, 1]` |
| `prev_address_months_count` | Months at previous address | `[−1, 380]`; `−1` = missing |
| `current_address_months_count` | Months at current address | `[−1, 406]`; `−1` = missing |
| `customer_age` | Age in decade bins (e.g. 20 → 20–29) | decade bins |
| `days_since_request` | Days since application was submitted | `[0, 78]` |
| `intended_balcon_amount` | Initial transferred amount for the application | `[−1, 108]`; **not HKD** |
| `zip_count_4w` | Applications in the same ZIP over 4 weeks | `[1, 5767]` |
| `velocity_6h` | Avg applications/hour over last 6 hours | wide range |
| `velocity_24h` | Avg applications/hour over last 24 hours | wide range |
| `velocity_4w` | Avg applications/hour over last 4 weeks | wide range |
| `bank_branch_count_8w` | Applications at selected branch over 8 weeks | `[0, 2521]` |
| `date_of_birth_distinct_emails_4w` | Distinct emails sharing DOB over 4 weeks | `[0, 42]` |
| `credit_risk_score` | Internal application risk score | `[−176, 387]` |
| `bank_months_count` | Age of prior account (if any), in months | `[−1, 31]`; `−1` = missing |
| `proposed_credit_limit` | Proposed credit limit | `[200, 2000]`; **not HKD** |
| `session_length_in_minutes` | Session length on banking site | `[−1, 107]`; `−1` = missing |
| `device_distinct_emails_8w` | Distinct emails from device over 8 weeks | `[0, 3]` |
| `device_fraud_count` | Prior fraudulent apps on the same device | `[0, 1]` |

#### 2. How to read the orange / blue bars

The feature plots show **share within each class**, not share of the whole population. For each feature value:

- **orange**: of all *fraud* accounts, what percentage land in this bin or category?
- **blue**: of all *non-fraud* accounts, what percentage land in this bin or category?

Because fraud is rare (~1%), population-share bars would look almost identical to the blue series. Within-class percentages make class-conditional differences easier to see.

#### 3. Anonymized category codes (A\* / B\* / C\*)

Feedzai **label-encoded** these fields for privacy. Official meanings stop at “payment plan / housing / employment.” But this isn't very useful, so we hypothesized what these fields might mean using data-driven methods with reference to typical bank KYC taxonomies. 

**A\* — `payment_type` (credit payment plan)**

| Code | Share (Base) | Best-guess meaning | Confidence |
|------|-------------:|--------------------|------------|
| AA | ~26% | Plan linked to an initial deposit / transferred opening balance (`intended_balcon_amount` much higher than other plans) | Medium |
| AB | ~37% | Standard / default plan (modal class) | Low–medium |
| AC | ~25% | Higher-risk plan variant (highest fraud among common plans) | Low |
| AD | ~12% | Secondary common plan | Low |
| AE | ~0.03% | Niche / specialty plan | Low |

**B\* — `housing_status` (residential status)**

| Code | Share (Base) | Best-guess meaning | Confidence |
|------|-------------:|--------------------|------------|
| BA | ~17% | Owner / owner-occupier *or* high-fraud self-report (long tenure, high income, **highest fraud**) | Low–medium |
| BB | ~26% | Mortgage / buying | Low–medium |
| BC | ~37% | Rent / tenant (largest class; shorter address tenure) | Medium |
| BD | ~2.6% | Temporary / shared / other mid-tier | Low |
| BE | ~17% | Lives with parents / family (young; long tenure; lowest fraud) | Medium–high |
| BF | ~0.2% | Rare residual housing type | Low |
| BG | ~0.03% | Rare residual housing type | Low |

**C\* — `employment_status` (strongest inferences)**

| Code | Share (Base) | Best-guess meaning | Confidence |
|------|-------------:|--------------------|------------|
| CA | ~73% | Employed / salaried | High |
| CB | ~14% | Self-employed / business | Medium–high |
| CC | ~4% | Retired (age median ~50) | High |
| CD | ~3% | Unemployed / not working | Medium |
| CE | ~2% | Student (age median ~20) | High |
| CF | ~4% | Homemaker / not in labor force | Medium |
| CG | ~0.05% | Other / unspecified | Medium |

Use CA/CC/CE as soft labels in discussion if helpful; treat A\* and most B\* codes as opaque factors unless you state the confidence clearly.


In [ ]:
missing = df[all_feature_cols].replace(-1, np.nan).isna().mean().sort_values(ascending=False)
print("Top missing / -1 rates:")
print(missing.head(15))
fig, ax = plt.subplots(figsize=(10, 4))
missing.head(20).plot(kind="bar", ax=ax, title="Missing or -1 rate by feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
train_preview = df[df["month"].isin([0, 1, 2, 3, 4, 5])]
num_for_corr = [c for c in numeric_cols if train_preview[c].replace(-1, np.nan).notna().sum() > 0]
corr = train_preview[num_for_corr].replace(-1, np.nan).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Numeric feature correlation (train months 0–5, Base)")
plt.tight_layout()
plt.show()


#### Variant base rates
Finally, we look at the base rates for each variant, which would be our reference in asking "how many times does our model perform better than random guess?" 

That is:
$$
\begin{align*}
&\text{Random-guess baseline} = \pi \\ \\


&\text{Lift} = \frac{\mathrm{PR{-}AUC}}{\pi} \\ \\

&C_{\mathrm{unlock}} = w \cdot h_{\mathrm{unlock}},
\quad
C_{\mathrm{check}} = w \cdot h_{\mathrm{check}}
\end{align*}
$$

$$ \\
\pi = \text{prevalence rate},\quad
w = \text{hourly clerk rate},\quad 
h_{\mathrm{unlock}},\, h_{\mathrm{check}} = \text{hours per false alarm}.
$$

In [ ]:
base_rates = compute_variant_base_rates(CONFIG["data_dir"], include_alias=False)
print(base_rates.to_string(index=False))
base_rates_path = CONFIG["results_dir"] / "variant_base_rates.csv"
base_rates.to_csv(base_rates_path, index=False)
print("Exported:", base_rates_path)

# Amount proxy sanity (Base) — units are NOT HKD
proxy = COST_CONFIG.fraud_volume_proxy_col
if proxy in df.columns:
    prox = df[proxy].replace(-1, np.nan)
    print(f"\n{proxy} on Base (dataset_amount_units, not currency):")
    print(prox.describe())
    print("Fraud mean:", df.loc[df["fraud_bool"] == 1, proxy].replace(-1, np.nan).mean())
    print("Non-fraud mean:", df.loc[df["fraud_bool"] == 0, proxy].replace(-1, np.nan).mean())


## Chapter 2: Preprocessing 
We have the following data split (in months): 
- Train `0–5`
- Validate `6`
- Test `7`


### Notes
a. **Leakage-safe fitting**: BAFPreprocessor is fit on training months only. Validation and test months are transformed with those frozen statistics, so later months never influence imputation, scaling, or encoding.

b. **Missing values**: In BAF, -1 is a missing marker on several numeric fields (for example address tenure and session length). Those values are converted to missing before imputation.

c. **Numeric features**: Numeric columns are imputed with the training median, optionally passed through a Yeo–Johnson transform to reduce skew, then standardized with StandardScaler. Yeo–Johnson is optional and is ablated in Stage 2.

d. **Categorical features**: Categorical columns are imputed with the training mode, then one-hot encoded. Unknown categories at scoring time are ignored rather than causing errors.

*All features retained. No manual feature dropping is applied in the default pipeline. Every column except `fraud_bool` and `month` enters the model through this preprocessor.*

In [ ]:
split = TimeSplit()
preprocessor = BAFPreprocessor(use_yeo_johnson=CONFIG["use_yeo_johnson"])
train_df, valid_df, test_df = preprocessor.split_by_month(df, split)
preprocessor.fit(train_df)

X_train, y_train = preprocessor.transform_with_target(train_df)
X_valid, y_valid = preprocessor.transform_with_target(valid_df)
X_test, y_test = preprocessor.transform_with_target(test_df)
feature_names = preprocessor.get_feature_names()

features_all = np.vstack([X_train.values, X_valid.values, X_test.values])
labels_all = np.concatenate([y_train.values, y_valid.values, y_test.values])
months_all = np.concatenate([
    train_df["month"].values,
    valid_df["month"].values,
    test_df["month"].values,
])
offsets = {
    "train": (0, len(train_df)),
    "valid": (len(train_df), len(train_df) + len(valid_df)),
    "test": (len(train_df) + len(valid_df), len(features_all)),
}
print("Train/valid/test:", X_train.shape, X_valid.shape, X_test.shape)
print("Preprocessed feature dim:", X_train.shape[1])


## Chapter 3: Anchor XGBoost 
Before adding property or kNN graphs, we need a strong, simple reference. If graph features cannot beat this model on validation PR-AUC, the extra complexity is hard to justify.

That is why, for this section, we train a standard tabular XGBoost model on Base using only preprocessed application features. No graph edges or neighbor statistics are generated yet.

#### Training setup
The same month split applies: fit on months 0–5, tune on month 6, report on month 7. Features come from BAFPreprocessor (Section 2). By default, Yeo–Johnson is on and SMOTE is off (use_smote_stage1: False).

#### Model
XGBoost is trained with `eval_metric="aucpr"`, so optimization aligns with PR-AUC. Validation scores get Platt calibration, and the operating threshold is chosen on month 6 (best F1). Afterwards, we evaluate on month 7.

#### Outputs
The run saves the model, test scores, and metrics under *results/eda_matrix/anchor/*. Those scores feed Stage 1 comparison, false-alarm analysis, and later champion documentation.

In [ ]:
anchor_out = CONFIG["results_dir"] / "anchor"
anchor_report = train_vanilla(
    DATA_PATH,
    output_dir=anchor_out,
    use_yeo_johnson=CONFIG["use_yeo_johnson"],
    use_smote=CONFIG["use_smote_stage1"],
    prefer_gpu=True,
)
anchor_model = joblib.load(anchor_out / "model.pkl")
test_pred_df = pd.read_csv(anchor_out / "test_predictions.csv")
anchor_test_scores = test_pred_df["score"].values
print("Anchor test PR-AUC:", anchor_report["metrics"]["pr_auc"])


## Chapter 4: Feature Importances
This section interprets the Section 3 tabular model rather than training a new one. XGBoost’s built-in importances show which preprocessed features the trees rely on most, in which the **top 30** are plotted and exported to CSV.

SHAP adds direction. For a sample of test rows, it shows how each feature pushes the score toward fraud or non-fraud. 

Remember! Importance does not prove causality, and one-hot columns split credit across related categories.


In [ ]:
imp = pd.DataFrame({
    "feature": feature_names,
    "importance": anchor_model.feature_importances_,
}).sort_values("importance", ascending=False)
imp["importance_pct"] = imp["importance"] / imp["importance"].sum()
top_n = 30
fig, ax = plt.subplots(figsize=(10, 8))
imp.head(top_n).plot.barh(x="feature", y="importance", ax=ax, legend=False)
ax.invert_yaxis()
ax.set_title(f"Anchor XGBoost — top {top_n} feature importances (Base)")
plt.tight_layout()
plt.show()
imp_path = CONFIG["results_dir"] / "feature_importance_anchor.csv"
imp.to_csv(imp_path, index=False)
print("Exported:", imp_path)


In [ ]:
try:
    import shap
    shap_n = min(CONFIG["shap_sample_size"], len(X_test))
    shap_idx = np.random.choice(len(X_test), shap_n, replace=False)
    X_shap = X_test.iloc[shap_idx]
    explainer = shap.TreeExplainer(anchor_model)
    shap_values = explainer.shap_values(X_shap)
    shap.summary_plot(shap_values, X_shap, feature_names=feature_names, max_display=20, show=False)
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("SHAP skipped:", exc)


## Chapter 5: Temporal kNN Graph

Here, we link applications that look alike in feature space, but only to earlier months. Each app gets `SIMILAR` edges to up to k=20 past neighbors (cosine similarity on preprocessed features).

The rule `neighbor_month < query_month` prevents future information from leaking into the graph. The builder also asserts `temporal_violations == 0`, where any violation means the graph is invalid and should be fixed before continuing.

Neighbor features are aggregated into embeddings for Stage 1 scoring. Per notebook policy, kNN is used only for graph construction. Neighbor stats are not concatenated as extra tabular columns.

Heavy graph builds are cached under knn_cache/ so reruns skip recomputation.


In [ ]:
knn_cache_dir = CONFIG["results_dir"] / "knn_cache"
knn_cache_dir.mkdir(parents=True, exist_ok=True)
emb_path = knn_cache_dir / "knn_embeddings.npy"
edges_path = knn_cache_dir / "knn_edges.parquet"
stats_path = knn_cache_dir / "knn_stats.json"

knn_cfg = TemporalKNNConfig(
    k=CONFIG["knn_k"],
    metric=CONFIG["knn_metric"],
    mutual=CONFIG["knn_mutual"],
)
if emb_path.exists() and edges_path.exists() and stats_path.exists():
    knn_embeddings = np.load(emb_path)
    knn_edges = pd.read_parquet(edges_path)
    knn_stats = json.loads(stats_path.read_text())
    print("Loaded kNN cache from", knn_cache_dir)
else:
    knn_edges = build_temporal_knn_graph(features_all, months_all, labels_all, knn_cfg)
    knn_stats = temporal_knn_stats(knn_edges, labels_all, months_all)
    knn_embeddings = mean_neighbor_features(features_all, knn_edges, len(features_all))
    np.save(emb_path, knn_embeddings)
    knn_edges.to_parquet(edges_path, index=False)
    stats_path.write_text(json.dumps(knn_stats, indent=2))
    print("Saved kNN cache to", knn_cache_dir)

print("Temporal kNN stats:", json.dumps(knn_stats, indent=2))
assert knn_stats["temporal_violations"] == 0, "Temporal leakage in kNN graph!"
print("kNN embedding shape:", knn_embeddings.shape)


## Chapter 6: Property Graph

Unlike kNN (similarity-based), the property graph uses typed entities: each application connects to shared values such as `source`, `device_os`, `payment_type`, and `employment_status` via `HAS_*` edges. Risk numerics (`velocity_24h`, `device_fraud_count`) are binned into low/mid/high bands and linked similarly.

The result is a heterogeneous graph: application nodes plus entity nodes, with an edge list and summary stats (node/edge counts, etc.).

Embeddings are computed from this structure and passed to XGBoost in Stage 1. This graph invents structure BAF does not ship with. That is, it groups apps that share devices, channels, or employment codes, which may surface coordinated or repeat-pattern fraud.

In [ ]:
app_nodes, prop_edges, entity_nodes = build_property_graph(df)
prop_stats = property_graph_stats(app_nodes, prop_edges, labels_all)
print("Property graph stats:", json.dumps(prop_stats, indent=2))
prop_embeddings = property_graph_embeddings(features_all, prop_edges, len(features_all))
print("Property embedding shape:", prop_embeddings.shape)


## Chapter 7: Stage 1 Comparison (Base)
Three arms are evaluated with the same split and metrics: 
1.  tabular XGBoost only
2.  tabular + temporal kNN embeddings
3.  tabular + property-graph embeddings

Each graph arm stacks original preprocessed features with graph embeddings, then trains XGBoost with the same hyperparameters and `eval_metric="aucpr"`. Scores are calibrated on month 6 and test PR-AUC is reported on month 7.

Results are saved to stage1_results.csv, with lift vs prevalence so performance is readable relative to the ~1.1% fraud base rate.

Decision gate: a graph arm should beat the tabular anchor on validation PR-AUC by a meaningful margin (e.g. ≥ 0.005) before treating graph enrichment as clearly worthwhile. The notebook may still lock the champion to the property graph for downstream stages even if tabular PR-AUC is slightly higher, as different runs sometimes lead to different results given the limited dataset. So do check the champion model.


In [ ]:
def train_graph_xgb(name, graph_features, use_smote=False):
    X_tr = np.hstack([X_train.values, graph_features[offsets["train"][0]:offsets["train"][1]]])
    X_va = np.hstack([X_valid.values, graph_features[offsets["valid"][0]:offsets["valid"][1]]])
    X_te = np.hstack([X_test.values, graph_features[offsets["test"][0]:offsets["test"][1]]])
    y_tr, y_va, y_te = y_train.values, y_valid.values, y_test.values
    if use_smote:
        from imblearn.over_sampling import SMOTE
        smote = SMOTE(sampling_strategy=0.5, random_state=CONFIG["seed"])
        X_tr, y_tr = smote.fit_resample(X_tr, y_tr)
    compute = resolve_xgb_compute(prefer_gpu=True)
    model = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=CONFIG["seed"],
        tree_method=compute["tree_method"],
        device=compute["device"],
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    valid_scores = model.predict_proba(X_va)[:, 1]
    test_scores = model.predict_proba(X_te)[:, 1]
    calibrator = calibrate_platt(y_va, valid_scores)
    test_cal = calibrator.model.predict_proba(test_scores.reshape(-1, 1))[:, 1]
    threshold = best_f1_threshold(y_va, valid_scores)
    metrics = evaluate_binary_classifier(y_te, test_scores, test_cal, threshold)
    return {"name": name, "metrics": metrics, "test_scores": test_scores, "model": model}


stage1_rows = [
    {"graph": "none", "model": "XGBoost_tabular", "metrics": anchor_report["metrics"], "test_scores": anchor_test_scores},
]
for label, emb in [("temporal_kNN", knn_embeddings), ("property", prop_embeddings)]:
    stage1_rows.append(train_graph_xgb(f"{label}_GNN_to_XGBoost", emb))

def _stage1_graph_label(row):
    if "graph" in row:
        return str(row["graph"])
    return str(row["name"]).replace("_GNN_to_XGBoost", "")


def _stage1_model_label(row):
    model = row.get("model", "GNN_to_XGBoost")
    return model if isinstance(model, str) else "GNN_to_XGBoost"


stage1_df = pd.DataFrame([
    {
        "graph": _stage1_graph_label(r),
        "model": _stage1_model_label(r),
        "pr_auc": r["metrics"]["pr_auc"],
        "recall": r["metrics"]["recall"],
        "brier_calibrated": r["metrics"]["brier_calibrated"],
        "alert_yield_pct": r["metrics"]["alert_yield_pct"],
    }
    for r in stage1_rows
])
print(stage1_df)
stage1_df.to_csv(CONFIG["results_dir"] / "stage1_results.csv", index=False)

# Attach Base lift vs prevalence
base_prev = float(base_rates.loc[base_rates["deck_label"] == "Base", "fraud_rate_overall"].iloc[0])
stage1_df["fraud_prevalence"] = base_prev
stage1_df["lift_vs_prevalence"] = stage1_df["pr_auc"] / base_prev
print("\nWith lift vs Base prevalence:")
print(stage1_df[["graph", "pr_auc", "fraud_prevalence", "lift_vs_prevalence"]])


## Chapter 8: Business Cost Model 
This section translates model scores into a *hypothetical* Hong Kong clerk-cost story.

Exposure per caught fraud uses `intended_balcon_amount` in dataset amount units (not HKD). Unlock and check costs use placeholder hourly rates and hours per false positive.

The model sweeps **BLOCK/ALERT** cutoffs and estimates a relative profit index: revenue from true positives minus labor from false positives, under stated assumptions. Optimal cutoffs and false-alarm summaries are exported.

This run uses anchor (tabular) test scores by default. The same analysis can be repeated for a Stage 1 graph winner if one clearly beats the baseline.

In [ ]:
exposure_tp = compute_exposure_tp_per_account(
    train_df,
    volume_col=COST_CONFIG.fraud_volume_proxy_col,
)
cost_unlock = cost_unlock_fp_per_account(COST_CONFIG)
cost_check = cost_check_fp_per_account(COST_CONFIG)

cost_assumptions = assumptions_dict(COST_CONFIG, exposure_tp, cost_unlock, cost_check)
assumptions_path = CONFIG["results_dir"] / "cost_assumptions.json"
assumptions_path.write_text(json.dumps(cost_assumptions, indent=2))
print(json.dumps(cost_assumptions, indent=2))


In [ ]:
cutoff_metrics = build_cutoff_metrics_table(
    y_test.values,
    anchor_test_scores,
    revenue_tp=exposure_tp,
    cost_unlock_fp=cost_unlock,
    cost_check_fp=cost_check,
)
cutoff_path = CONFIG["results_dir"] / "cutoff_economics.csv"
cutoff_metrics.to_csv(cutoff_path, index=False)

optimal = maximize_profit(cutoff_metrics, cost_check_fp=cost_check, config=COST_CONFIG)
# Prefer relative_profit_index key for exports
if "max_profit" in optimal and "expected_profit_hkd" not in optimal:
    optimal["relative_profit_index"] = optimal.get("relative_profit_index", optimal["max_profit"])
optimal_path = CONFIG["results_dir"] / "optimal_cutoffs.json"
optimal_path.write_text(json.dumps(optimal, indent=2))

# False-alarm summary at F1, alert, and block cutoffs
valid_scores_anchor = joblib.load(anchor_out / "model.pkl").predict_proba(X_valid)[:, 1]
f1_cut = float(best_f1_threshold(y_valid.values, valid_scores_anchor))
fa_cutoffs = {
    "f1_on_valid": f1_cut,
    "block": float(optimal["block_cutoff"]),
    "alert": float(optimal["alert_cutoff"]),
}
fa_summary = build_false_alarm_summary(
    y_test.values,
    anchor_test_scores,
    exposure_tp,
    cost_unlock,
    cost_check,
    fa_cutoffs,
    config=COST_CONFIG,
)
fa_path = CONFIG["results_dir"] / "false_alarm_summary.csv"
fa_summary.to_csv(fa_path, index=False)
(CONFIG["results_dir"] / "false_alarm_summary.json").write_text(
    json.dumps(fa_summary.to_dict(orient="records"), indent=2)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_cutoff_economics(
    cutoff_metrics,
    ax=axes[0],
    ops_currency=COST_CONFIG.ops_currency,
    exposure_unit=COST_CONFIG.exposure_unit,
)
plot_false_alarm_rate(cutoff_metrics, ax=axes[1])
plt.tight_layout()
plt.show()

print("Optimal cutoffs:", json.dumps(optimal, indent=2))
print("False-alarm summary:")
print(fa_summary.to_string(index=False))
print("Exported:", cutoff_path, optimal_path, fa_path)


## Chapter 9: Stage Gates: Stages 2 and 3

Stage 2 (primary) grid-searches Yeo–Johnson on/off × SMOTE on/off on Variant I. Variant I keeps fraud prevalence near Base (~1.1%) but stresses group-size imbalance (90%/10%), so preprocessing effects are easier to read than on variants with different prevalence or drift.

Stage 2 (reference) optionally repeats the same grid on Base for comparison.

Stage 3 freezes the champion preprocessing settings and retrains on Base + Variants I–V to test generalization across bias types (prevalence shift, separability, temporal drift, etc.). `variant_6` is excluded from unique claims — it is a Base alias only.

Full preprocess×SMOTE grids on Variants II–V are intentionally avoided: they are expensive and confound which preprocessing choice actually generalizes.


In [ ]:
# Stage 2 — primary ablation on Variant I
RUN_STAGE2 = True
RUN_STAGE2_BASE_REFERENCE = True  # optional Base reference table
stage2_results = []

def run_preprocess_grid(data_path: Path, arm_label: str):
    rows = []
    for use_yj in [True, False]:
        for use_smote in [False, True]:
            report = train_vanilla(
                data_path,
                output_dir=CONFIG["results_dir"] / f"{arm_label}_yj{use_yj}_smote{use_smote}",
                use_yeo_johnson=use_yj,
                use_smote=use_smote,
            )
            rows.append({
                "arm": arm_label,
                "deck_label": "Variant I" if arm_label == "variant_I" else "Base",
                "use_yeo_johnson": use_yj,
                "use_smote": use_smote,
                "pr_auc": report["metrics"]["pr_auc"],
                "recall": report["metrics"]["recall"],
            })
    return rows

if RUN_STAGE2:
    v1 = Path(CONFIG["stage2_ablation_variant"])
    if not v1.exists():
        raise FileNotFoundError(f"Variant I CSV missing: {v1}")
    stage2_results.extend(run_preprocess_grid(v1, "variant_I"))
    stage2_df = pd.DataFrame(stage2_results)
    stage2_df.to_csv(CONFIG["results_dir"] / "stage2_ablations_variant_I.csv", index=False)
    print("Variant I Stage 2:")
    print(stage2_df)
else:
    print("Stage 2 (Variant I) skipped — set RUN_STAGE2 = True after Stage 1 gate.")

if RUN_STAGE2_BASE_REFERENCE:
    base_ref = run_preprocess_grid(DATA_PATH, "base_reference")
    pd.DataFrame(base_ref).to_csv(CONFIG["results_dir"] / "stage2_ablations_base_reference.csv", index=False)
    print("Base reference Stage 2 written.")


In [ ]:
# Stage 3 — Base + Variants I–V only (no unique claim for variant_6)
RUN_STAGE3 = True
variant_rows = []

if RUN_STAGE3:
    for deck_label, vpath in CONFIG["eval_suite"]:
        if not vpath.exists():
            print("Skip missing:", deck_label, vpath)
            continue
        report = train_vanilla(
            vpath,
            output_dir=CONFIG["results_dir"] / f"stage4_{vpath.stem}",
            use_yeo_johnson=CONFIG["use_yeo_johnson"],
            use_smote=CONFIG["use_smote_stage1"],
        )
        variant_rows.append({
            "deck_label": deck_label,
            "file": vpath.name,
            "pr_auc": report["metrics"]["pr_auc"],
            "recall": report["metrics"]["recall"],
        })
    variant_df = pd.DataFrame(variant_rows)
    # Join base rates for lift
    br = compute_variant_base_rates(CONFIG["data_dir"], include_alias=False)
    variant_df = variant_df.merge(
        br[["deck_label", "fraud_rate_overall"]],
        on="deck_label",
        how="left",
    )
    variant_df["lift_vs_prevalence"] = variant_df["pr_auc"] / variant_df["fraud_rate_overall"]
    variant_df.to_csv(CONFIG["results_dir"] / "stage4_variants.csv", index=False)
    print(variant_df)
    print(variant_df.describe())
else:
    print("Stage 3 skipped — enable RUN_STAGE3 after champion is frozen.")
    print("Eval suite (Base + I–V):", [(l, str(p)) for l, p in CONFIG["eval_suite"]])


## Chapter 10: Conclusions and Exports
This section consolidates everything into files under `results/eda_matrix/` for slides, reports, or downstream pipelines.

Exports include graph stats (kNN + property), cost assumptions, false-alarm summary, Stage 1/2/3 CSVs, feature importances, and champion_config.json.

The champion record documents the property graph arm under property_graph_locked policy, alongside the PR-AUC leader for transparency. Lift tables join variant base rates with model PR-AUC so readers can compare performance vs chance across Base and I–V.

If no graph arm beats the tabular anchor, that is still a valid negative result. Graphs did not pay off under this protocol. Cost outputs remain unit-honest placeholders, not measured savings from BAF.

The printed lift table is deck-ready: prevalence, PR-AUC, and lift per official variant label.

In [ ]:
# Refresh base rates with Stage 1 Base PR-AUC for deck table draft
prop_rows = stage1_df[stage1_df["graph"] == "property"]
if prop_rows.empty:
    raise ValueError("Stage 1 champion is locked to property graph, but no property row was found.")
best_row = prop_rows.iloc[0]
pr_auc_leader = stage1_df.sort_values("pr_auc", ascending=False).iloc[0]
pr_map = {"Base": float(best_row["pr_auc"])}
lift_table = attach_lift(base_rates, pr_map)
lift_table.to_csv(CONFIG["results_dir"] / "variant_base_rates_with_lift.csv", index=False)

graph_stats = {
    "temporal_knn": knn_stats,
    "property_graph": prop_stats,
    "knn_policy": "graph_edges_only_no_tabular_enrichment",
    "baf_suite": "Base + Variants I–V; variant_6 is Base alias only",
    "n_raw_features": len(all_feature_cols),
    "n_preprocessed_features": len(feature_names),
    "config": {k: str(v) for k, v in CONFIG.items()},
    "cost_assumptions": cost_assumptions,
}
(CONFIG["results_dir"] / "graph_stats.json").write_text(json.dumps(graph_stats, indent=2))

champion = {
    "graph": best_row["graph"],
    "model": best_row["model"],
    "pr_auc_test": float(best_row["pr_auc"]),
    "champion_policy": "property_graph_locked",
    "pr_auc_leader_graph": str(pr_auc_leader["graph"]),
    "pr_auc_leader": float(pr_auc_leader["pr_auc"]),
    "lift_vs_base_prevalence": float(best_row["lift_vs_prevalence"]),
    "stage1_complete": True,
    "stage2_complete": bool(RUN_STAGE2),
    "stage3_complete": bool(RUN_STAGE3),
    "optimal_block_cutoff": optimal["block_cutoff"],
    "optimal_alert_cutoff": optimal["alert_cutoff"],
    "relative_profit_index": optimal.get("relative_profit_index", optimal.get("max_profit")),
    "cost_disclaimer": COST_DISCLAIMER,
}
(CONFIG["results_dir"] / "champion_config.json").write_text(json.dumps(champion, indent=2))

print("Exported:", CONFIG["results_dir"])
print("Champion:", champion)
print("\nDeck-ready lift table (Base PR-AUC attached; fill I–V after Stage 3):")
print(lift_table[["deck_label", "fraud_rate_overall", "literature_fraud_rate_approx", "pr_auc", "lift_vs_prevalence"]].to_string(index=False))
